In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/07_fact_dq_quarantine_log
# Purpose  : Governance table summarizing daily rejected record counts per source
#            table and rejection reason, enabling time-series Data Quality monitoring.
#
# Key Technical Note:
#   Implements explicit event-time Watermarking for stateful aggregation (`count()`)
#   over streaming data. Mandatory to purge old window state and maintain Append Mode
#   compatibility for DLT Streaming Tables.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_dq_quarantine_log",
    comment="Governance fact - daily count of rejected records per source table and rule",
)
def fact_dq_quarantine_log():
    encounters_q = dlt.read_stream("covid19_lakehouse.quarantine.encounters")
    conditions_q = dlt.read_stream("covid19_lakehouse.quarantine.conditions")
    medications_q = dlt.read_stream("covid19_lakehouse.quarantine.medications")
    careplans_q = dlt.read_stream("covid19_lakehouse.quarantine.careplans")
    allergies_q = dlt.read_stream("covid19_lakehouse.quarantine.allergies")
    devices_q = dlt.read_stream("covid19_lakehouse.quarantine.devices")

    all_quarantined = (
        encounters_q.unionByName(conditions_q)
        .unionByName(medications_q)
        .unionByName(careplans_q)
        .unionByName(allergies_q)
        .unionByName(devices_q)
    )

    return (
        all_quarantined
        .withWatermark("quarantined_at", "1 day")
        .groupBy(
            F.window(F.col("quarantined_at"), "1 day").alias("batch_window"),
            F.col("source_table"),
            F.col("rule_violated"),
        )
        .count()
        .withColumn("window_start", F.col("batch_window.start"))  # Pre-extracted - clearer for readability
        .select(
            F.xxhash64("source_table", "rule_violated", "window_start").alias("quarantine_key"),
            F.col("source_table"),
            F.col("rule_violated"),
            F.col("count").alias("rejected_row_count"),
            F.date_format(F.col("window_start"), "yyyyMMdd").cast("int").alias("batch_date_key"),
            F.current_timestamp().alias("logged_at"),
        )
    )